In [ ]:
import pandas as pd
import numpy as np
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier

In [2]:
X_train = pd.read_csv('../data/x_train_clean.csv')
X_train = X_train.drop(columns=['id'])
y_train = pd.read_csv('../data/y_train_clean.csv')
y_train = y_train['target']
X_test = pd.read_csv('../data/x_test_clean.csv')
X_test = X_test.drop(columns=['id'])

In [4]:


#modèle 1
weights_full = compute_sample_weight('balanced', y_train)
model_xgb = XGBClassifier(
    objective='multi:softprob',
    num_class=4,
    random_state=42
)
model_xgb.fit(X_train, y_train, sample_weight=weights_full)

#modèle 2
model_rf = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=4,
    max_features='sqrt'
)
model_rf.fit(X_train, y_train)

#modèle 3
model_cat = CatBoostClassifier(
    loss_function='MultiClass',
    random_seed=42,
    auto_class_weights='Balanced',
    iterations=400,
    depth=8,
    learning_rate=0.03,
    l2_leaf_reg=1,
    bagging_temperature=1,
    grow_policy='SymmetricTree'
)
model_cat.fit(X_train, y_train)


0:	learn: 1.3785768	total: 169ms	remaining: 1m 7s
1:	learn: 1.3702185	total: 333ms	remaining: 1m 6s
2:	learn: 1.3622378	total: 500ms	remaining: 1m 6s
3:	learn: 1.3556766	total: 650ms	remaining: 1m 4s
4:	learn: 1.3492461	total: 841ms	remaining: 1m 6s
5:	learn: 1.3427269	total: 1.06s	remaining: 1m 9s
6:	learn: 1.3362427	total: 1.32s	remaining: 1m 13s
7:	learn: 1.3310174	total: 1.56s	remaining: 1m 16s
8:	learn: 1.3255058	total: 1.81s	remaining: 1m 18s
9:	learn: 1.3180127	total: 2.08s	remaining: 1m 21s
10:	learn: 1.3123460	total: 2.32s	remaining: 1m 22s
11:	learn: 1.3071511	total: 2.55s	remaining: 1m 22s
12:	learn: 1.3019276	total: 2.96s	remaining: 1m 28s
13:	learn: 1.2964697	total: 3.18s	remaining: 1m 27s
14:	learn: 1.2914678	total: 3.46s	remaining: 1m 28s
15:	learn: 1.2863750	total: 3.73s	remaining: 1m 29s
16:	learn: 1.2811875	total: 3.97s	remaining: 1m 29s
17:	learn: 1.2766318	total: 4.25s	remaining: 1m 30s
18:	learn: 1.2728513	total: 4.5s	remaining: 1m 30s
19:	learn: 1.2669484	total: 4

CatBoostClassifier(auto_class_weights='Balanced', bagging_temperature=1, depth=8, grow_policy='SymmetricTree', iterations=400, l2_leaf_reg=1, learning_rate=0.03, loss_function='MultiClass', random_seed=42)

In [8]:
#soft voting
prob_cat = model_cat.predict_proba(X_test)
prob_xgb = model_xgb.predict_proba(X_test)
prob_rf = model_rf.predict_proba(X_test)

prob_ensemble = (0.4 * prob_cat) + (0.3 * prob_xgb) + (0.3 * prob_rf)
predictions_num = np.argmax(prob_ensemble, axis=1)

In [9]:
postes = {0: 'Assistant', 1: 'Executive', 2: 'Manager', 3: 'Director'}
predictions_labels = [postes[p] for p in predictions_num]

# créer le fichier
test_ids = pd.read_csv('../data/x_test_clean.csv', usecols=['id'])['id']
predictions_final = pd.DataFrame({
    'id': test_ids,
    'position': predictions_labels
})
predictions_final.to_csv('../results/predictions_ensemble_1.csv', index=True)
